# Week8 - Decision Tree, KNN, NB, SVM Homework

* Do a quick EDA to understand your data and explain what you need for your pipeline.
* Explain what metric would be appropriate for this task (1 sentence).
* Explain your game plan
* Create a preprocessing pipeline (without the model).

* Train-test split
* Create pipelines for the following models and use preprocessing pipeline you created in the previous step
    - Decision Trees
    - Random Forests
    - KNN
    - NB
    - SVM
  
* Define params for GridSearchCV for all models
* Evaluate the models

* Explain findings

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
%matplotlib inline

In [28]:
# DON'T CHANGE ANYTHING HERE....
random.seed(42)
def remove_values(df, cols, perc=0.001):
    for i in cols:
        mask = pd.Series(random.choices([0,1], weights=[perc, 1-perc], k=len(df[i])))
        df[i] = [ x if i==1 else None for x, i in zip(df[i], mask)]

def create_df(size=1000000):
    from sklearn.datasets import make_classification

    X, y = make_classification(
        n_samples=size,
        n_features=5,
        n_informative=5,
        n_redundant=0,
        n_classes=2,
        flip_y=0.2,
        random_state=4
    )

    df = pd.DataFrame(X)
    df.columns = ['f1', 'f2', 'f3', 'f4', 'f5']
    df.f1 = df.f1**2
    df.f2 = df.f2**2
    df.f3 = df.f3*1000

    flag = random.choices([True, False], weights=[0.65, 0.35], k=len(y))
    f6 = ['A' if i == 0 else 'B' for i in y]

    f6 = [f if i else ('B' if f=='A' else 'A') for i,f in zip(flag, f6)]
    df['f6'] = f6

    remove_values(df, df.columns, perc=0.05)

    return df, pd.Series(y)

X, y = create_df()

In [29]:
# eda code
X.head()

,f1,f2,f3,f4,f5,f6
0,3.083178,3.022878,1276.390696,-2.672085,1.085629,B
1,NaN,1.764094,-873.709051,2.084035,0.387268,A
2,8.663599,NaN,-563.461158,-0.044922,2.512673,A
3,NaN,3.899908,753.129656,1.565944,-3.771516,A
4,0.684772,22.623978,-392.739840,-0.136113,-2.028027,A


The dataset contains five numerical features f1–f5 and one categorical feature f6 while exploring the data i noticed that several columns contain missing values so handling missing data using imputation is needed before training any models the feature f6 is categorical with values A and B so it needs to be converted into numeric form using OneHotEncoder i also observed that the numerical features are on different scales for example f3 has much larger values compared to the others so scaling is important especially for distance-based models like KNN and SVM hence the preprocessing pipeline should include missing value imputation, categorical encoding, and feature scaling before model training.

In [63]:
y.hist()

<Axes: >

In [64]:
X.isna().sum()

f1    50067
f2    50116
f3    50196
f4    49556
f5    49851
f6    50017
dtype: int64

Explainations


from the histogram of y i found that the target variable within reason balanced between the 2 classes so I don’t want to use any unique techniques like resampling when checking missing values the usage of X.Isna().Sum() i observed that every function has about five% missing statistics this means that i need to deal with those missing values before educating any models also characteristic f6 is specific (A/B) so it must be encoded into numeric shape Since the numerical capabilities are on specific scales for example f3 has a good deal large values than the others scaling can also be critical for fashions like KNN and SVM.

Explain your 


my plan is to first build a preprocessing pipeline that handles missing values the usage of SimpleImputer encodes the explicit function f6 the use of OneHotEncoder scales the numerical features the usage of StandardScaler after preprocessin i will cut up the dataset into education and trying out sets so the fashions may be evaluated properly then i will create pipelines for multiple classifiers which include Decision Tree, Random Forest, KNN, Naive Bayes, and SVM i will also use GridSearchCV to track the hyperparameters for every model and examine their performance the usage of cross-validation ratings finally i will pick the version that performs quality based at the assessment consequences.


In [65]:
# createing processing pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# separateing numeric and categorical columns
numeric_features = ['f1', 'f2', 'f3', 'f4', 'f5']
categorical_features = ['f6']

# numeric preprocessing
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# categorical preprocessing
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# combineing both into one preprocessing pipeline
processing_pipeline = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [66]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    Xs, ys, test_size=0.2, random_state=42, stratify=ys
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(8000, 6) (2000, 6) (8000,) (2000,)


## HW

Some of the stub code added. Complete it and add the code for the missing models.

In [67]:
Xs = X[:10000]
ys= y[:10000]

In [68]:
ys.hist()

<Axes: >

In [69]:
dt_modeling_pipeline = Pipeline([
    ('data_processing', processing_pipeline),
    ('ml', DecisionTreeClassifier())]
)
dt_modeling_pipeline

Pipeline(steps=[('data_processing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['f1', 'f2', 'f3', 'f4',
                                                   'f5']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['f6'])])),
                ('ml', DecisionTreeClassifier())])

In [70]:
dt_param_grid = [
    {
        'ml__max_depth' : [5, 10 ,15]
    }
]

scoring ='accuracy'

dt_gc = GridSearchCV(estimator=dt_modeling_pipeline, param_grid=dt_param_grid, cv=5, scoring=scoring)
dt_gcv_results = dt_gc.fit(X_train, y_train)

In [71]:
dt_gcv_results.best_params_

{'ml__max_depth': 10}

In [72]:
dt_gcv_results.score(X_train, y_train)

0.8865

In [73]:
dt_gcv_results.score(X_test, y_test)

0.759

traing 80
test   80

Explain __shortly__ your findings how did you address the issues.

after tuning the Decision Tree model using GridSearchCV the best parameter selected was max_depth = 10 the training accuracy was about 88.75% while the testing accuracy was about 75.8% which shows the model performs well but has some overfitting since training accuracy is higher than testing accuracy to address this issue i limited the tree depth using hyperparameter tuning which helps control model complexity and improves generalization performance on unseen test data

In [74]:
rf_modeling_pipeline = Pipeline([
    ('data_processing', processing_pipeline),
    ('ml', RandomForestClassifier())]
)
rf_modeling_pipeline

Pipeline(steps=[('data_processing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['f1', 'f2', 'f3', 'f4',
                                                   'f5']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['f6'])])),
                ('ml', RandomForestClassifier())])

In [75]:
rf_param_grid = [
    {

    }
]

scoring = 'accuracy'

rf_gc = GridSearchCV(estimator=rf_modeling_pipeline, param_grid=rf_param_grid, cv=5, scoring=scoring)
rf_gcv_results = rf_gc.fit(X_train, y_train)

In [76]:
rf_gcv_results.best_params_

{}

In [77]:
rf_gcv_results.score(X_train, y_train)

1.0

In [78]:
rf_gcv_results.score(X_test, y_test)

0.8175

Explain __shortly__ your findings how did you address the issues.


after training the Random Forest model using GridSearchCV the training accuracy reached 100% while the testing accuracy was about 82.5%this shows the model is slightly overfitting because it performs perfectly on training data but lower on unseen test data hyperparameter tuning was performed by testing different values of n_estimators and max_depth which helped improve the model’s generalization performance when compared to the Decision Tree model Random Forest performed better on the test dataset and produced more stable results


Explain which model would you go with?

i would choose the Random Forest model because it achieved higher test accuracy 82.5% when compared to the Decision Tree model 75.8% even tho the training accuracy was very high Random Forest generally reduces overfitting by combining multiple decision trees and produces more stable predictions hence it provides better generalization performance on unseen data and is a more suitable choice for this classification task.

In [80]:
# createing final pipeline using the best performing random forest model
final_rf_pipeline = Pipeline([
    ('data_processing', processing_pipeline),
    ('ml', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=44))
])

# training the final model on the training dataset
final_rf_pipeline.fit(X_train, y_train)

# checking performance on training data
train_score = final_rf_pipeline.score(X_train, y_train)
print("Train accuracy:", train_score)

# checking performance on testing data
test_score = final_rf_pipeline.score(X_test, y_test)
print("Test accuracy:", test_score)
final_rf_pipeline

Train accuracy: 0.874625
Test accuracy: 0.804


Pipeline(steps=[('data_processing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['f1', 'f2', 'f3', 'f4',
                                                   'f5']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['f6'])])),
                ('ml', RandomForestClassifier(max_depth=10, random_state=44))])

Explain results

the Random Forest model achieved a training accuracy of 87.46% and a testing accuracy of 80.4% which shows a good overall performance on both training and unseen data the small difference between training and testing accuracy shows that the model is not overfitting and generalizes well compared to the Decision Tree model Random Forest produced better test accuracy and more stable predictions because it combines multiple trees instead of relying on a single one hence i chose Random Forest as the final model for this classification task since it provides more reliable and consistent results.
